# Práctica 2 — PyCaret (Profesor)

## Flujo reproducible y sin fuga

El `setup` reserva un holdout reproducible del 20 %. PyCaret compara y ajusta mediante CV de 5 folds sobre el 80 % de entrenamiento. El holdout se evalúa una sola vez, al final.


## 1. Carga del dataset

El CSV no se versiona en el repositorio. El profesorado debe entregarlo y colocarlo en la ruta indicada en la guía antes de ejecutar este notebook. La carga falla de forma explícita si falta: no sustituyas el dataset docente por datos sintéticos ni por el CSV bruto de Lab1.


In [ ]:
from pathlib import Path
import pandas as pd
from pycaret.classification import compare_models, finalize_model, predict_model, pull, setup, tune_model

def find_project_root() -> Path:
    """Locate the repository from the current Jupyter working directory."""
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pixi.toml").is_file() and (candidate / "03-machine-learning").is_dir():
            return candidate
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del repositorio. Abre el notebook dentro del repositorio PIA."
    )


PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "03-machine-learning/03-practicas/midterm-capstone/Peliculas/Practica_Peliculas_OK/data/movies.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"No se encuentra {DATA_PATH}. Solicita el movies.csv validado y guárdalo en esa ruta."
    )
df = pd.read_csv(DATA_PATH)
df.head()


## Configuración

No se escalan ni codifican los datos antes de `setup`: PyCaret ajusta sus transformadores sobre entrenamiento dentro de su pipeline.


In [ ]:
setup(
    data=df,
    target='rating_high',
    train_size=0.8,
    session_id=42,
    fold=5,
)


## Comparación y ajuste: solo entrenamiento

`compare_models` y `tune_model` usan CV de 5 folds; el holdout queda reservado. Se prioriza `F1` porque equilibra precisión y exhaustividad en una clasificación potencialmente desbalanceada. Si el caso de uso penaliza más falsos positivos o falsos negativos, se debe justificar otra métrica y emplearla de forma consistente en `sort` y `optimize`.


In [ ]:
metric = 'F1'
best_model = compare_models(sort=metric, fold=5)
comparison_results = pull()
comparison_results.head()
selected_model = best_model


In [ ]:
# El ajuste es opcional; con False, `selected_model` sigue siendo el mejor candidato de CV.
tune = True
selected_model = best_model
if tune:
    tuned_model = tune_model(best_model, optimize=metric, fold=5)
    tuning_results = pull()
    display(tuning_results)
    selected_model = tuned_model


## Evaluación final: holdout reservado

Esta celda se ejecuta una vez, una vez cerrada la decisión. No se llama a `finalize_model` antes: ese paso incorporaría el holdout y solo correspondería después de documentar la evaluación final, si se fuera a desplegar. `finalize_model(selected_model)` no genera una nueva evaluación válida y no sustituye las métricas del holdout.


In [ ]:
holdout_predictions = predict_model(selected_model)
holdout_metrics = pull()
holdout_metrics

# Solo después de documentar el holdout y si se prepara el despliegue:
# final_model = finalize_model(selected_model)


In [ ]:
holdout_predictions.head()


## Evidencia docente mínima

Revisar en el notebook: configuración (`train_size=0.8`, `session_id=42`, `fold=5`), tabla de CV de comparación, ajustes aplicados y tabla de métricas/predicciones del holdout. Las métricas dependen del dataset y no se fijan en esta plantilla.
